In [ ]:
#------------------------------------------------ Import Lib ----------------------------------------
import re
import os
import datetime
import requests
import pandas as pd
from bs4 import BeautifulSoup

import urllib3
urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)


#------------------------------------------------ Begin_ fileName ----------------------------------------

regulatorName = 'KG NBKR' ## change to current controller name

print(f"Running {regulatorName} Web Scraping Tool v.1.0")

now = datetime.datetime.now()

filename = '{} SQL Ready {}.xlsx'.format(regulatorName, str(now).replace(":", ".")[:-7])

try:
    scriptfolder = os.path.dirname(os.path.abspath(__file__))  ## production environment (.py)
except NameError:
    scriptfolder = os.getcwd()  ## notebook environment

os.chdir(scriptfolder)

tempfolder = os.path.join(scriptfolder, 'tempfolder')  # output folder

if os.path.exists(tempfolder):
    for rem in os.listdir(tempfolder):
        os.remove(os.path.join(tempfolder, rem))
else:
    os.mkdir(tempfolder)

In [ ]:
#------------------------------------------------ Begin_Variable ----------------------------------------
sqldict={'bvdid': [], 'priority': [], 'ListLabel': [], 'Typology': [], 'EntryType': [], 'Name': [], 'InternalID_1': [], 'InternalID_1_type': [], 'InternalID_2': [], 
          'InternalID_2_type': [], 'InternalID_3': [], 'InternalID_3_type': [], 'CoType': [], 'License_Type': [], 'Address_1': [], 'Address_2': [], 'City': [], 
          'Zip': [], 'Cntry': [], 'Phone': [], 'Fax': [], 'Website': [], 'Email': [], 'RegulationType': [], 'RegulationTypeCode': [], 'RegulationDate': [], 'CancellationDate': [], 
          'RegCtry': [], 'RegCode' : [], 'ListCode': [], 'ListLanguage': [], 'ListValidityDate': [], 'ListName': [], 'ListProcessDate': [], 'LEI Code': [], 'BIC SWIFT Code': [], 'Name - Mother Company': [],
          'Address_1 - Mother company': [], 'Address_2 -  Mother company': [], 'City - Mother company': [], 'Zip - Mother company': [], 'Cntry - Mother company': [], 
          'Phone - Mother company': []}


regdict={

        regulatorName+' 1': 'https://www.nbkr.kg/index1.jsp?item=71&lang=ENG',

        }


Typology={

       regulatorName + ' 1': 'List of commercial banks of the Kyrgyz Republic',

        }


processdate = now.strftime('%Y-%m-%d')

HEADERS = {'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/126.0.0.0 Safari/537.36'}

In [ ]:
#------------------------------------------------ Begin_Fouction ----------------------------------------

def bourange_same_length_array(sqldict) :

    maxlen = len(sqldict['ListProcessDate'])

    for key, val in sqldict.items():

        if len(sqldict[key]) != maxlen:

            empty = []

            total_empty = maxlen - len(sqldict[key])

            for i in range(total_empty):

                empty.append('')

            sqldict[key]=sqldict[key]+empty

    return sqldict


# The English page mixes visually-identical Cyrillic letters into Latin words
# (e.g. 'Оpen', 'Сompany', 'Тhe oреn ... соmраnу'). Map them back to Latin.
CYR2LAT = {'А':'A','В':'B','С':'C','Е':'E','Н':'H','К':'K','М':'M','О':'O','Р':'P','Т':'T','Х':'X',
           'а':'a','с':'c','е':'e','о':'o','р':'p','у':'y','х':'x'}


def clean_text(text):
    s = text.replace('\xa0', ' ')
    s = re.sub(r'\s+', ' ', s).strip()
    return s


def fix_homoglyphs(text):
    return ''.join(CYR2LAT.get(ch, ch) for ch in text)


def clean_name(text):
    s = fix_homoglyphs(clean_text(text))
    s = s.replace('« ', '«').replace(' »', '»')  # tidy spaces inside guillemets
    return s.strip()


def parse_address(text):
    # cell looks like: '720021, Bishkek, #101, Shopokov str., ph: 33 30 00' (optional 'fax: ...')
    s = clean_text(text)
    fax_ = ''
    m = re.search(r'fax[:.\s]*([\d][\d\s,\-]*)', s, re.I)
    if m:
        fax_ = m.group(1).strip(' ,.')
        s = s[:m.start()] + s[m.end():]
    phone_ = ''
    m = re.search(r'\bph[:.\s]*([\d][\d\s,\-]*)', s, re.I)
    if m:
        phone_ = m.group(1).strip(' ,.')
        s = s[:m.start()] + s[m.end():]
    s = fix_homoglyphs(s).strip(' ,.')
    zip_ = ''
    m = re.match(r'^([A-Za-z]?\d{4,6})\b[\s,]*', s)
    if m:
        zip_ = m.group(1)
        s = s[m.end():]
    city_ = ''
    parts = [p.strip() for p in s.split(',') if p.strip(' .')]
    if parts and not re.search(r'[\d#]', parts[0]):
        city_ = parts[0]
        parts = parts[1:]
    addr_ = ', '.join(parts).strip(' ,')
    return zip_, city_, addr_, phone_, fax_


def parse_contact(text):
    # cell looks like: 'bishkek@kkb.kg www.kkb.kg' (sometimes several e-mails, sometimes 'www. site.kg')
    s = clean_text(text)
    emails = re.findall(r'[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}', s)
    s = s.replace('www. ', 'www.')
    m = re.search(r'(https?://\S+|www\.[\w.\-/]+)', s)
    website_ = m.group(1).strip(' ,') if m else ''
    return ', '.join(emails), website_

In [ ]:
#------------------------------------------------ Begin_Main ----------------------------------------

for k, reg in enumerate(regdict):
    # print(f"[INFO] : Working {k+1}/{len(regdict)} _({reg})_ ")
    if reg == 'KG NBKR 1':
        resp = requests.get(regdict[reg], headers=HEADERS, verify=False, timeout=60)
        resp.raise_for_status()
        soup = BeautifulSoup(resp.text, 'lxml')

        # the page holds several lists; only take the table right after this heading
        heading = soup.find(string=re.compile(
            r'List\s+of\s+commercial\s+banks\s+of\s+the\s+Kyrgyz\s+Republic\s+and\s+number\s+of\s+their\s+branches', re.I))
        if heading is None:
            raise RuntimeError('KG NBKR 1: target list heading not found - page layout may have changed')
        table_ = heading.find_next('table')
        if table_ is None:
            raise RuntimeError('KG NBKR 1: no table found after the target list heading')

        for row in table_.find_all('tr'):
            cells = row.find_all(['td', 'th'])
            if len(cells) < 7:
                continue
            texts = [clean_text(c.get_text(' ', strip=True)) for c in cells]
            if 'full name' in texts[1].lower():  # header row
                continue
            name_ = clean_name(texts[1])
            if not name_:
                continue
            # columns: N | full name | abbreviated name | address+phone | chairman | nr of branches | e-mail/web
            zip_, city_, address_, phone_, fax_ = parse_address(texts[3])
            email_, website_ = parse_contact(texts[6])

            sqldict['Name'].append(name_)
            sqldict['Address_1'].append(address_)
            sqldict['City'].append(city_)
            sqldict['Zip'].append(zip_)
            sqldict['Phone'].append(phone_)
            sqldict['Fax'].append(fax_)
            sqldict['Email'].append(email_)
            sqldict['Website'].append(website_)
            sqldict['ListProcessDate'].append(processdate)
            sqldict['ListName'].append(Typology[reg])
            sqldict['RegCtry'].append(reg.split()[0])
            sqldict['RegCode'].append(reg.split()[1])
            sqldict['ListCode'].append(reg.split()[2])
            sqldict['RegulationType'].append('Regulated')

        sqldict = bourange_same_length_array(sqldict)

In [ ]:
#------------------------------------------------ Begin_writer and save df to excel  ----------------------------------------
os.chdir(scriptfolder)

df=pd.DataFrame(sqldict)

df = df[df['Name']!='']

df.to_excel(os.path.join(tempfolder, filename), sheet_name='SQL Ready', index=False)

print(f"[INFO] : Saved {len(df)} rows to {os.path.join(tempfolder, filename)}")